# RNN Encoder-Decoder for Statistical Machine Translation

Paper:  
- [Learning Phrase Representations using RNN Encoder-Decoder for Statistical Machine Translation](https://emnlp2014.org/papers/pdf/EMNLP2014179.pdf)


**Requirements**

In [8]:
from __future__ import unicode_literals, print_function, division
from io import open
import unicodedata
import re
import random

import torch
import torch.nn as nn
from torch import optim
import torch.nn.functional as F

import numpy as np
from torch.utils.data import TensorDataset, DataLoader, RandomSampler

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

torch.set_default_device(device)
print(f"Using device = {torch.get_default_device()}")

Using device = cpu


## Loading Data

Download from the following link: 
- [https://www.manythings.org/anki/](https://download.pytorch.org/tutorial/data.zip)

Then, extract the data: `data/eng-fra.txt`. The file is a tab separated list of translation pairs. 

<p align="center">
  <img src="./images/DataPreparation.drawio.png" alt="Data Preparation">
</p>

- To make a word list from sentence, let's create a Class the sentence/line and creates `word2index`, `index2word`, and `word2count`.

In [9]:
SOS_token = 0 # Start of the Sentence
EOS_token = 1 # End of the Sentence

class Lang:
    def __init__(self, name):
        self.name = name
        self.word2index = {}
        self.word2count = {}
        self.index2word = {0: "SOS", 1: "EOS"}
        self.n_words = 2  # Count SOS and EOS

    def addSentence(self, sentence):
        for word in sentence.split(' '):
            self.addWord(word)

    def addWord(self, word):
        if word not in self.word2index:
            self.word2index[word] = self.n_words
            self.word2count[word] = 1
            self.index2word[self.n_words] = word
            self.n_words += 1
        else:
            self.word2count[word] += 1

- The input the the `addSentence` method that is required to make the `word2index`, `index2word`, and `word2count` is (obviously) a sentence. 
- Therefore, so let's create a method to 
    - read the Dataset/file, 
    - split it into lines and then 
    - create sentence pairs (Language1 Sentence, Equivalent Language2 Sentence) 
    - Normalize.

In [10]:
# Turn a Unicode string to plain ASCII, thanks to
# https://stackoverflow.com/a/518232/2809427
def unicodeToAscii(s):
    return ''.join(
        c for c in unicodedata.normalize('NFD', s)
        if unicodedata.category(c) != 'Mn'
    )

# Lowercase, trim, and remove non-letter characters
def normalizeString(s):
    s = unicodeToAscii(s.lower().strip())
    s = re.sub(r"([.!?])", r" \1", s)
    s = re.sub(r"[^a-zA-Z!?]+", r" ", s)
    return s.strip()

In [11]:
def readLangs(path:str):
    lang1 = 'eng'; lang2 = 'fra'
    print("Reading lines...")

    # Read the file and split into lines
    lines = open(path, encoding='utf-8').\
        read().strip().split('\n')

    # Split every line into pairs and normalize (english to french)
    pairs = [[normalizeString(s) for s in l.split('\t')] for l in lines]

    # Reverse pairs: English-French -> French-English
    pairs = [list(reversed(p)) for p in pairs]

    # Input is French, output is English
    input_lang = Lang(lang2)
    output_lang = Lang(lang1)

    return input_lang, output_lang, pairs

Since there are a lot of example sentences and we want to train something quickly, we’ll trim the data set to only relatively short and simple sentences. Here the maximum length is 10 words (that includes ending punctuation) and we’re filtering to sentences that translate to the form “I am” or “He is” etc. (accounting for apostrophes replaced earlier).

The overall purpose is to clean and prepare sentence pairs for training a language model by removing unsuitable examples.

In [26]:
MAX_LENGTH = 5

eng_prefixes = (
    "i am ", "i m ",
    "he is", "he s ",
    "she is", "she s ",
    "you are", "you re ",
    "we are", "we re ",
    "they are", "they re "
)

def filterPair(p):
    return len(p[0].split(' ')) < MAX_LENGTH and \
        len(p[1].split(' ')) < MAX_LENGTH and \
        p[1].startswith(eng_prefixes)


def filterPairs(pairs):
    return [pair for pair in pairs if filterPair(pair)]

Now, let's bundle everything up as per the diagram above: 

In [13]:
def prepareData(path):
    input_lang, output_lang, pairs = readLangs(path)
    print("Read %s sentence pairs" % len(pairs))
    pairs = filterPairs(pairs)
    print("Trimmed to %s sentence pairs" % len(pairs))
    print("Counting words...")
    for pair in pairs:
        input_lang.addSentence(pair[0])
        output_lang.addSentence(pair[1])
    print("Counted words:")
    print(input_lang.name, input_lang.n_words)
    print(output_lang.name, output_lang.n_words)
    return input_lang, output_lang, pairs

In [14]:
PATH = 'data/eng-fra.txt'

input_lang, output_lang, pairs = prepareData(PATH)
print(random.choice(pairs))

output_lang.word2index['am']  # try different English words. 

Reading lines...
Read 135842 sentence pairs
Trimmed to 3272 sentence pairs
Counting words...
Counted words:
fra 1757
eng 967
['vous etes imprudent', 'you re foolish']


15

`Note`: Here, we have only inserted lower case in the word2index, therefore, use of uppercase letters will yield an error. 

## Encoder-Decoder Network

A Sequence to Sequence network, or seq2seq network, or Encoder-Decoder network, is a model consisting of two RNNs called the encoder and decoder. The encoder reads an input sequence and outputs a single vector, and the decoder reads that vector to produce an output sequence.

### Encoder

- Encoder is a RNN, that process the input one word at a time. 
- However, since `Encoder` doesn't need to produce an output, we only need the hidden state after processing the whole input sequence/sentence. Refer to the [paper](https://emnlp2014.org/papers/pdf/EMNLP2014179.pdf) for confirmation. 

In [15]:
class EncoderRNN(nn.Module):
    def __init__(self, input_size, hidden_size, dropout_p=0.1):
        super(EncoderRNN, self).__init__()
        self.hidden_size = hidden_size

        self.embedding = nn.Embedding(input_size, hidden_size)
        self.rnn = nn.RNN(hidden_size, hidden_size, batch_first=True)
        self.dropout = nn.Dropout(dropout_p)

    def forward(self, input):
        embedded = self.dropout(self.embedding(input))
        output, hidden = self.rnn(embedded)
        return output, hidden

Here, we convert the input word indices into dense embedding vectors. During training, these embeddings are learned and gradually capture the semantic relationships and meanings of the words.


### Decoder

- Decoder in another RNN. 
- It takes the encoder output vector and outputs a sequence of words.
- Decoder we use only last output of the encoder.
- The last output/hidden state is called the `context vector`.
    - which encodes context from the entire sequence. 
-  This context vector is used as the initial hidden state of the decoder.
- The initial input token is the start-of-string `<SOS>` token

In [16]:
class DecoderRNN(nn.Module):
    def __init__(self, hidden_size, output_size):
        super(DecoderRNN, self).__init__()
        self.embedding = nn.Embedding(output_size, hidden_size)
        self.rnn = nn.RNN(hidden_size, hidden_size, batch_first=True)
        self.out = nn.Linear(hidden_size, output_size)

    def forward(self, encoder_outputs, encoder_hidden, target_tensor=None):
        batch_size = encoder_outputs.size(0)
        decoder_input = torch.empty(batch_size, 1, dtype=torch.long, device=device).fill_(SOS_token)
        decoder_hidden = encoder_hidden
        decoder_outputs = []

        for i in range(MAX_LENGTH):
            decoder_output, decoder_hidden  = self.forward_step(decoder_input, decoder_hidden)
            decoder_outputs.append(decoder_output)

            if target_tensor is not None:
                # Teacher forcing: Feed the target as the next input
                decoder_input = target_tensor[:, i].unsqueeze(1) # Teacher forcing
            else:
                # Without teacher forcing: use its own predictions as the next input
                _, topi = decoder_output.topk(1) # values, index of the highest-scoring word
                decoder_input = topi.squeeze(-1).detach()  # detach from history as input (removes the last dimension before detaching)

        decoder_outputs = torch.cat(decoder_outputs, dim=1)
        decoder_outputs = F.log_softmax(decoder_outputs, dim=-1)
        return decoder_outputs, decoder_hidden, None # We return `None` for consistency in the training loop

    def forward_step(self, input, hidden):
        output = self.embedding(input)
        output = F.relu(output)
        output, hidden = self.rnn(output, hidden)
        output = self.out(output)
        return output, hidden

**Without Teacher Forcing/During Inference:**
- We need the index instead of the value/tensor because, embedding layer requires the index not the tensor. 

**Why detach():**
- Normally, PyTorch tracks every operation to build the computation graph for backpropagation.
<div style="text-align:center; font-family:monospace; font-size:18px;">
    <div>decoder step 1</div>
    <div style="font-size:30px;">↓</div>
    <div style="font-weight:bold;">prediction</div>
    <div style="font-size:30px;">↓</div>
    <div>decoder step 2</div>
    <div style="font-size:30px;">↓</div>
    <div style="font-weight:bold;">prediction</div>
</div>

- Without detaching, PyTorch would try to backpropagate through this entire generation chain.
- Instead we say:
    - Treat this predicted token as a new input. Do not keep the previous computation history.

**MAX_LENGTH** 
- refers to the maximum length of the output/predicted sentence, not the input sentence.

**ENCODER vs DECODER**

| Aspect                                    | Encoder RNN                       | Decoder RNN                                                            |
| ----------------------------------------- | ------------------------------------- | -------------------------------------------------------------------------- |
| Purpose                                   | Read and understand an input sequence | Generate an output sequence                                                |
| Example                                   | Input: "I love cats"                  | Output: "J'aime les chats"                                                 |
| Do we know the whole sequence beforehand? | Yes                                   | No                                                                         |
| Where is the loop?                        | Inside `nn.RNN`            | In your `for i in range(MAX_LENGTH)` loop                                  |
| Code pattern                              | `output, hidden = self.rnn(sequence)` | `for i in range(MAX_LENGTH): output, hidden = self.rnn(one_token, hidden)` |
| Number of RNN calls                       | Usually 1 call                        | Usually `MAX_LENGTH` calls                                                 |
| Input to RNN                              | Entire sequence                       | One token at a time                                                        |
| Sequence length given to RNN              | `seq_len = input sentence length`     | `seq_len = 1` per call                                                     |
| Hidden state recurrence                   | Automatic                             | Automatic                                                                  |
| Output recurrence                         | Usually none                          | Manually created                                                           |
| Previous hidden state comes from          | Previous input time step              | Previous decoder step                                                      |
| Previous output becomes next input?       | No                                    | Yes (without teacher forcing)                                              |


**Code Comparision:**

| **Component**               | **Encoder**                           | **Decoder**                                   |
| --------------------------- | ------------------------------------- | --------------------------------------------- |
| Calling the recurrent layer | `output, hidden = self.rnn(embedded)` | `output, hidden = self.rnn(input, hidden)`    |
| Number of calls             | One call processes the whole sequence | One call per generated token                  |
| Loop location               | Inside PyTorch                        | Inside your `for i in range(MAX_LENGTH)` loop |
| Recurrent unit sees         | All input tokens                      | One generated token at each step              |
| Generates tokens?           | No                                    | Yes                                           |


## Training



## Prepare Training Data

For each pair:
-  we will need an input tensor (indexes of the words in the input sentence) and 
- target tensor (indexes of the words in the target sentence). 
- While creating these vectors we will append the EOS token to both sequences.

In [17]:
def indexesFromSentence(lang, sentence):
    return [lang.word2index[word] for word in sentence.split(' ')]

def tensorFromSentence(lang, sentence):
    indexes = indexesFromSentence(lang, sentence)
    indexes.append(EOS_token)
    return torch.tensor(indexes, dtype=torch.long, device=device).view(1, -1)

def tensorsFromPair(pair):
    input_tensor = tensorFromSentence(input_lang, pair[0])
    target_tensor = tensorFromSentence(output_lang, pair[1])
    return (input_tensor, target_tensor)

def get_dataloader(batch_size):
    input_lang, output_lang, pairs = prepareData(path=PATH)

    n = len(pairs)
    input_ids = np.zeros((n, MAX_LENGTH), dtype=np.int32)
    target_ids = np.zeros((n, MAX_LENGTH), dtype=np.int32)

    for idx, (inp, tgt) in enumerate(pairs):
        inp_ids = indexesFromSentence(input_lang, inp)
        tgt_ids = indexesFromSentence(output_lang, tgt)
        inp_ids.append(EOS_token)
        tgt_ids.append(EOS_token)
        input_ids[idx, :len(inp_ids)] = inp_ids
        target_ids[idx, :len(tgt_ids)] = tgt_ids

    train_data = TensorDataset(torch.LongTensor(input_ids).to(device),
                               torch.LongTensor(target_ids).to(device))

    train_sampler = RandomSampler(train_data)
    train_dataloader = DataLoader(train_data, sampler=train_sampler, batch_size=batch_size)
    return input_lang, output_lang, train_dataloader

### Training Loop

In [18]:
def train_epoch(dataloader, encoder, decoder, encoder_optimizer,
          decoder_optimizer, criterion):

    total_loss = 0
    for data in dataloader:
        input_tensor, target_tensor = data

        encoder_optimizer.zero_grad()
        decoder_optimizer.zero_grad()

        encoder_outputs, encoder_hidden = encoder(input_tensor)
        decoder_outputs, _, _ = decoder(encoder_outputs, encoder_hidden, target_tensor) # using teacher forcing

        loss = criterion(
            decoder_outputs.view(-1, decoder_outputs.size(-1)),
            target_tensor.view(-1)
        )
        loss.backward()

        encoder_optimizer.step()
        decoder_optimizer.step()

        total_loss += loss.item()

    return total_loss / len(dataloader)

In [19]:
import time
import math

def asMinutes(s):
    m = math.floor(s / 60)
    s -= m * 60
    return '%dm %ds' % (m, s)

def timeSince(since, percent):
    now = time.time()
    s = now - since
    es = s / (percent)
    rs = es - s
    return '%s (- %s)' % (asMinutes(s), asMinutes(rs))

In [20]:
import matplotlib.pyplot as plt
plt.switch_backend('agg')
import matplotlib.ticker as ticker
import numpy as np

def showPlot(points):
    plt.figure()
    fig, ax = plt.subplots()
    # this locator puts ticks at regular intervals
    loc = ticker.MultipleLocator(base=0.2)
    ax.yaxis.set_major_locator(loc)
    plt.plot(points)

In [21]:
def train(train_dataloader, encoder, decoder, n_epochs, learning_rate=0.001,
               print_every=100, plot_every=100):
    start = time.time()
    plot_losses = []
    print_loss_total = 0  # Reset every print_every
    plot_loss_total = 0  # Reset every plot_every

    encoder_optimizer = optim.Adam(encoder.parameters(), lr=learning_rate)
    decoder_optimizer = optim.Adam(decoder.parameters(), lr=learning_rate)
    criterion = nn.NLLLoss()

    for epoch in range(1, n_epochs + 1):
        loss = train_epoch(train_dataloader, encoder, decoder, encoder_optimizer, decoder_optimizer, criterion)
        print_loss_total += loss
        plot_loss_total += loss

        if epoch % print_every == 0:
            print_loss_avg = print_loss_total / print_every
            print_loss_total = 0
            print('%s (%d %d%%) %.4f' % (timeSince(start, epoch / n_epochs),
                                        epoch, epoch / n_epochs * 100, print_loss_avg))

        if epoch % plot_every == 0:
            plot_loss_avg = plot_loss_total / plot_every
            plot_losses.append(plot_loss_avg)
            plot_loss_total = 0

    showPlot(plot_losses)

## Evaluation Code

In [22]:
def evaluate(encoder, decoder, sentence, input_lang, output_lang):
    with torch.no_grad():
        input_tensor = tensorFromSentence(input_lang, sentence)

        encoder_outputs, encoder_hidden = encoder(input_tensor)
        decoder_outputs, decoder_hidden, decoder_attn = decoder(encoder_outputs, encoder_hidden)

        _, topi = decoder_outputs.topk(1)
        decoded_ids = topi.squeeze()

        decoded_words = []
        for idx in decoded_ids:
            if idx.item() == EOS_token:
                decoded_words.append('<EOS>')
                break
            decoded_words.append(output_lang.index2word[idx.item()])
    return decoded_words, decoder_attn

In [23]:
def evaluateRandomly(encoder, decoder, n=10):
    for i in range(n):
        pair = random.choice(pairs)
        print('>', pair[0])
        print('=', pair[1])
        output_words, _ = evaluate(encoder, decoder, pair[0], input_lang, output_lang)
        output_sentence = ' '.join(output_words)
        print('<', output_sentence)
        print('')

### Training and Evaluating

In [24]:
hidden_size = 128
batch_size = 32
EPOCHS = 200

input_lang, output_lang, train_dataloader = get_dataloader(batch_size)

encoder = EncoderRNN(input_lang.n_words, hidden_size).to(device)
decoder = DecoderRNN(hidden_size, output_lang.n_words).to(device)

train(train_dataloader, encoder, decoder, EPOCHS, print_every=5, plot_every=5)

Reading lines...
Read 135842 sentence pairs
Trimmed to 3272 sentence pairs
Counting words...
Counted words:
fra 1757
eng 967
0m 6s (- 3m 59s) (5 2%) 1.9620
0m 10s (- 3m 27s) (10 5%) 1.2690
0m 15s (- 3m 14s) (15 7%) 1.0533
0m 20s (- 3m 6s) (20 10%) 0.9081
0m 26s (- 3m 2s) (25 12%) 0.7868
0m 31s (- 2m 57s) (30 15%) 0.6923
0m 36s (- 2m 50s) (35 17%) 0.6130
0m 40s (- 2m 43s) (40 20%) 0.5435
0m 45s (- 2m 37s) (45 22%) 0.4850
0m 50s (- 2m 31s) (50 25%) 0.4300
0m 55s (- 2m 25s) (55 27%) 0.3809
0m 59s (- 2m 19s) (60 30%) 0.3394
1m 4s (- 2m 14s) (65 32%) 0.2992
1m 9s (- 2m 8s) (70 35%) 0.2671
1m 14s (- 2m 3s) (75 37%) 0.2366
1m 18s (- 1m 58s) (80 40%) 0.2120
1m 23s (- 1m 52s) (85 42%) 0.1874
1m 28s (- 1m 47s) (90 45%) 0.1705
1m 33s (- 1m 42s) (95 47%) 0.1520
1m 37s (- 1m 37s) (100 50%) 0.1398
1m 42s (- 1m 32s) (105 52%) 0.1275
1m 47s (- 1m 27s) (110 55%) 0.1154
1m 51s (- 1m 22s) (115 57%) 0.1101
1m 56s (- 1m 17s) (120 60%) 0.1026
2m 1s (- 1m 12s) (125 62%) 0.0946
2m 6s (- 1m 7s) (130 65%) 0.088

In [25]:
encoder.eval()
decoder.eval()
evaluateRandomly(encoder, decoder)

> vous etes tres courageuses
= you are very brave
< you re very brave <EOS>

> il est trop saoul
= he s too drunk
< he s too drunk <EOS>

> j etudie la musique
= i m learning music
< i m learning music <EOS>

> nous sommes occupes
= we re busy
< we re all prisoners <EOS>

> vous etes trop maigrichon
= you re too skinny
< you re too skinny <EOS>

> vous etes acariatre
= you re cantankerous
< you re hurting him <EOS>

> tu es fort occupe
= you re very busy
< you re very busy <EOS>

> elle joue a monopoly
= she s playing monopoly
< she s playing monopoly <EOS>

> tu es ambitieuse
= you re ambitious
< you re completely delusional <EOS>

> je cuis du pain
= i am baking bread
< i am baking bread <EOS>



`Note`: Could train this with varying degree of success. 

# Lab 7 (B1): Encoder-Decoder Models & Word Embeddings Report

## 1. Objective
The objective of this lab is to understand how Word Embeddings overcome the limitations of One-Hot Vectors and to implement an Encoder-Decoder (Sequence-to-Sequence) model from scratch in PyTorch to translate English sentences into French.

## 2. Methodology
* **Data Preparation:** An English-to-French dataset is loaded. Sentences are filtered for simplicity, normalized, and tokenized. A Vocabulary (`word2index`) is generated to map words to integer indices.
* **Encoder Architecture:** An `EncoderRNN` is built using an `nn.Embedding` layer (to convert sparse word indices into dense semantic vectors) and an `nn.RNN` layer to process the sequence. It compresses the entire English sentence into a final hidden state (Context Vector).
* **Decoder Architecture:** A `DecoderRNN` is constructed. It uses the Encoder's Context Vector as its initial state and autoregressively generates the French sentence word-by-word.
* **Training:** The model is trained using Negative Log Likelihood Loss (`NLLLoss`) and the Adam optimizer. Teacher Forcing is employed during training to speed up convergence by feeding the true previous target word instead of the model's prediction.

## 3. Results
The model successfully trains and the loss steadily decreases. During evaluation (inference), the model accurately translates simple English sentences into French by sequentially generating words until the `<EOS>` token is predicted.

## 4. Discussion
**Why do One-Hot Vectors fail in NLP, and how do Embeddings solve it?**

The one-hot vectors are entirely orthogonal (equal distance between every word mathematically), which means that there is zero knowledge of semantics in them. The vectors are also very sparse, taking up too much memory especially when the vocabulary size is very large.
This problem is solved by dense word embeddings where the words are represented using floating point vectors. In training, words appearing in a similar context are pulled close together through mathematical calculation.

## 5. Conclusion
Through this experiment, it is shown that RNNs can be developed further to become powerful Sequence-to-Sequence models by separating the network into two parts called the Encoder and the Decoder. This solves problems involving varying lengths of input and output sequences, such as in Machine Translation, where Word Embedding creates a semantically richer language representation than one-hot encoding.